# Digital Buck Converter Control

Run the whole project in your browser. Nothing is installed on your machine and
no account is needed beyond a Google login.

Press **Runtime -> Run all**, or run the cells one at a time.

Repository: https://github.com/fatinnihal532-hub/digital-buck-control


## Setup


In [ ]:
!git clone -q https://github.com/fatinnihal532-hub/digital-buck-control.git
%cd digital-buck-control
!pip install -q -r requirements.txt
print('ready')


## 1. Does the model agree with theory, and with itself?

Seventeen checks. Some compare the simulation against closed-form expressions;
others check that the compensator actually achieves the crossover and phase
margin it was designed for; the last one runs the averaged model and the
switching model through the same load step and requires them to agree.

Takes about twenty seconds.


In [ ]:
!python3 verify.py


## 2. Design a compensator

Pick a crossover frequency and a phase margin. The k-factor method places the
poles and zeros, Tustin discretises it, and `b` and `a` are the difference-equation
coefficients you would type into firmware.

Try `f_cross=4000, pm_deg=50` first, then try `600` and watch what happens.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
from models.buck_blocks import BuckParams
from models.design import design, to_discrete, margins, closed_loop_poles

F_CROSS = 4000      # Hz   <-- change me
PM      = 50        # deg  <-- change me
ORDER   = 3         # 2 = type II, 3 = type III

p = BuckParams(n_dpwm=12)
(num, den), rec = design(p, f_cross=F_CROSS, pm_deg=PM, order=ORDER)
b, a = to_discrete(num, den, p.ts, f_prewarp=F_CROSS)
mg = margins(p, num, den)
cl = closed_loop_poles(p, b, a)

print(f'plant: LC corner {p.f_lc:.0f} Hz, Q = {p.q_lc:.1f}, ESR zero {p.f_esr/1e3:.0f} kHz')
print(f'zeros at {rec["f_zero"]:.0f} Hz, poles at {rec["f_pole"]:.0f} Hz')
print(f'achieved crossover {mg["f_cross"]:.0f} Hz, phase margin {mg["pm"]:.1f} deg, gain margin {mg["gm_db"]:.1f} dB')
print(f'0 dB crossings: {mg["n_crossings"]}')
print()
print(f'largest closed-loop pole |z| = {cl["spectral_radius"]:.4f}  ->  {"STABLE" if cl["stable"] else "UNSTABLE"}')
print('b =', b.round(6))
print('a =', a.round(6))


### Why the pole magnitude is the test, not the phase margin

This output filter has a Q of 6. Its resonant peak can push the loop gain back
above unity after it has already fallen through, so the magnitude crosses 0 dB
three times. Phase margin is defined at one crossing and says nothing about the
other two.

Design for 600 Hz above and you get a compensator with a plausible-looking Bode
plot, a reported phase margin of about -40 degrees, and a closed-loop pole at
1.0018 -- just outside the unit circle. It grows 0.18 % per sample, so a short
simulation looks settled and a long one does not. Run the next cell to see that.


In [ ]:
from models.buck_model import simulate
import numpy as np

pp = lambda x: float(np.max(x) - np.min(x))
for label, order, fc, pm in (('type II, 200 Hz', 2, 200, 60),
                             ('type II, 600 Hz', 2, 600, 60),
                             ('type III, 4 kHz', 3, 4000, 50)):
    (n_, d_), _ = design(p, f_cross=fc, pm_deg=pm, order=order)
    bb, aa = to_discrete(n_, d_, p.ts, f_prewarp=fc)
    rho = closed_loop_poles(p, bb, aa)['spectral_radius']
    r = simulate(p, bb, aa, t_end=60e-3, dt=1e-6, t_ramp=1e-3, averaged=True)
    early = r['vout'][(r['t'] > 5e-3) & (r['t'] <= 10e-3)]
    late  = r['vout'][(r['t'] > 55e-3)]
    print(f'{label:16s} max|z|={rho:.4f}  '
          f'swing at 5-10 ms {pp(early):6.3f} V   at 55-60 ms {pp(late):6.3f} V')


## 3. What the choice costs at the output

Start-up on a reference ramp, then a 2 A to 4 A load step.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))
for label, order, fc, pm in (('type II, 200 Hz', 2, 200, 60),
                             ('type III, 4 kHz', 3, 4000, 50)):
    (n_, d_), _ = design(p, f_cross=fc, pm_deg=pm, order=order)
    bb, aa = to_discrete(n_, d_, p.ts, f_prewarp=fc)
    r = simulate(p, bb, aa, t_end=14e-3, dt=4e-7, t_ramp=1e-3,
                 load_step=(6e-3, 3.0), averaged=True)
    ax.plot(r['t']*1e3, r['vout'], lw=1.6, label=label)
ax.axhline(12, lw=.8, ls='--', color='grey')
ax.set_xlabel('time (ms)'); ax.set_ylabel('$V_{out}$ (V)'); ax.set_ylim(0, 20)
ax.set_title('Start-up, then a load step at 6 ms'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()


## 4. Rebuild every figure in the README

About forty-five seconds.


In [ ]:
!python3 make_figures.py


In [ ]:
from IPython.display import SVG, display
for name in ['loop_bode', 'pole_map', 'transient', 'ripple',
             'antiwindup', 'limit_cycle']:
    display(SVG(filename=f'figures/{name}_light.svg'))


---

Built by Fatin Nihal Islam. The full write-up, including what the model
deliberately leaves out, is in the
[repository README](https://github.com/fatinnihal532-hub/digital-buck-control).
